# Which Nomis / ONS-API datasets can we pull directly at ITL2?

Scans two API catalogues and flags which datasets are published at **ITL2** — indicators we could pull at our region level directly (no LA aggregation), refreshably.

- **Part A — Nomis** (`nomisweb.co.uk/api`, ~1,600 datasets): the main one. For each dataset, lists its geography types and checks for **ITL2 (as of Jan 2025)** (`TYPE420`). For datasets that have it, finds **which periods actually hold ITL2 data** — this matters: Nomis often lists the ITL2 type but only fills recent periods (e.g. APS at ITL2 only from 2025).
- **Part B — ONS beta API** (`api.beta.ons.gov.uk`, ~340 datasets): checks each dataset's geography options for ITL2 codes (`TLC3`-style).

Outputs: `docs/nomis_itl2_scan.csv` and `docs/ons_api_itl2_scan.csv`. A full run takes a few minutes (requests run in parallel, a few at a time).

**Caveats.**
- *Nomis periods with data* are found by bisection between the first and latest period, assuming ITL2 data, once it starts, continues to the latest period. A period counts as "has data" if any value is non-missing in the first 25,000 rows returned (other dimensions left at their defaults = all categories).
- Nomis also carries **older NUTS2 boundary sets** (NUTS 2003/2010/2013/2016), which sometimes hold the history `TYPE420` lacks. They are listed in `nuts2_types` for information only: NUTS2 regions (37–41) are **not** ITL2 2025 regions (46), so they can't be used without a boundary crosswalk.
- Both scans only cover what is *on* these APIs; ONS also publishes some ITL2 tables only as spreadsheets (e.g. regional GVA/GDHI).

In [1]:
import io, json, re, time, urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import pandas as pd

BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
OUT_NOMIS = BASE / "docs" / "nomis_itl2_scan.csv"
OUT_ONS   = BASE / "docs" / "ons_api_itl2_scan.csv"
NOMIS = "https://www.nomisweb.co.uk/api/v01/dataset"
ONS   = "https://api.beta.ons.gov.uk/v1"

WORKERS   = 6                      # parallel requests for Nomis
ONS_WORKERS = 2                    # the ONS beta API rate-limits (HTTP 429), so go slower there
ITL2_TYPE = "TYPE420"              # Nomis: international territorial levels - level 2 (as of Jan 2025)
ITL3_TYPE = "TYPE421"
LAD_TYPE  = "TYPE424"              # Nomis: local authorities: district / unitary (as of April 2023)

def get(url, tries=6):
    for a in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "regional-panel-itl2-scan"})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if a == tries - 1 or e.code not in (429, 500, 502, 503, 504): raise
            wait = e.headers.get("Retry-After")                     # honour rate-limit back-off
            time.sleep(float(wait) if wait and wait.isdigit() else 5 * (a + 1))
        except Exception:
            if a == tries - 1: raise
            time.sleep(2 * (a + 1))

def get_json(url):
    return json.loads(get(url))

def pmap(fn, items, workers=WORKERS):
    """Run fn over items in parallel, printing progress; returns results in order."""
    out, n = [], len(items)
    with ThreadPoolExecutor(workers) as ex:
        for i, r in enumerate(ex.map(fn, items), 1):
            out.append(r)
            if i % 50 == 0 or i == n: print(f"  {i}/{n}", end="\r")
    print()
    return out

# Part A — Nomis

## A1. Catalogue

One request returns every dataset with its name and metadata (status, last updated).

In [2]:
kf = get_json(f"{NOMIS}/def.sdmx.json")["structure"]["keyfamilies"]["keyfamily"]
def ann(k, title):
    return next((a["annotationtext"] for a in k.get("annotations", {}).get("annotation", [])
                 if a["annotationtitle"] == title), None)
nomis_cat = pd.DataFrame([{"id": k["id"], "name": k["name"]["value"], "status": ann(k, "Status"),
                           "last_updated": (ann(k, "LastUpdated") or "")[:10]} for k in kf])
print(f"Nomis datasets: {len(nomis_cat)}")
print(nomis_cat.status.value_counts().to_string())

Nomis datasets: 1617
status
Current (being actively updated)           1180
Historical (not actively being updated)     437


## A2. Geography types per dataset

Requesting the geography code-list for the bare type `TYPE` returns the list of geography types the dataset offers (one cheap request each).

In [3]:
def geo_types(ds):
    try:
        lists = (get_json(f"{NOMIS}/{ds}/geography/TYPE.def.sdmx.json").get("structure") or {}).get("codelists")
        codes = (lists or {}).get("codelist", [{}])[0].get("code", [])
        if not codes:                                        # e.g. census flow / origin-destination tables
            return {}, "no geography types listed"
        return {c["value"]: c["description"]["value"] for c in codes}, ""
    except Exception as e:
        return {}, f"error: {e}"

t0 = time.time()
types = pmap(geo_types, nomis_cat.id.tolist())
nomis_cat["n_geo_types"]  = [len(t) for t, _ in types]
nomis_cat["itl2_type"]    = [ITL2_TYPE in t for t, _ in types]
nomis_cat["itl3_type"]    = [ITL3_TYPE in t for t, _ in types]
nomis_cat["lad_2023"]     = [LAD_TYPE in t for t, _ in types]
nomis_cat["nuts2_types"]  = ["; ".join(v for v in t.values() if re.search(r"nuts \d{4} level 2", v)) for t, _ in types]
nomis_cat["note"]         = [n for _, n in types]
print(f"done in {time.time()-t0:.0f}s | notes: {nomis_cat.note[nomis_cat.note != ''].str[:40].value_counts().to_dict()}")
print(f"Datasets offering the ITL2 (Jan 2025) type: {nomis_cat.itl2_type.sum()} | "
      f"LA (April 2023): {nomis_cat.lad_2023.sum()}")

  1617/1617
done in 50s | notes: {'no geography types listed': 46}
Datasets offering the ITL2 (Jan 2025) type: 12 | LA (April 2023): 318


## A3. Which periods actually hold ITL2 data?

For each dataset with the ITL2 type: get its periods, find the latest period with ITL2 data (checking back up to 3 periods), then bisect for the earliest.

In [4]:
def has_itl2_data(ds, period):
    raw = get(f"{NOMIS}/{ds}.data.csv?geography={ITL2_TYPE}&date={period}&select=obs_value&recordlimit=25000")
    if not raw.strip():
        return False
    return pd.read_csv(io.BytesIO(raw)).OBS_VALUE.notna().any()

def itl2_coverage(ds):
    rec = {"id": ds, "n_periods": None, "itl2_first": None, "itl2_last": None, "itl2_n_periods": 0, "itl2_note": ""}
    try:
        periods = [c["value"] for c in get_json(f"{NOMIS}/{ds}/time.def.sdmx.json")["structure"]["codelists"]["codelist"][0]["code"]]
        rec["n_periods"] = len(periods)
        last = next((i for i in range(len(periods) - 1, max(len(periods) - 4, -1), -1) if has_itl2_data(ds, periods[i])), None)
        if last is None:
            rec["itl2_note"] = "no ITL2 data in latest periods"
            return rec
        lo, hi = 0, last                      # earliest period with data is in [lo, hi]
        while lo < hi:
            mid = (lo + hi) // 2
            if has_itl2_data(ds, periods[mid]): hi = mid
            else: lo = mid + 1
        rec.update(itl2_first=periods[lo], itl2_last=periods[last], itl2_n_periods=last - lo + 1)
    except Exception as e:
        rec["itl2_note"] = f"error: {e}"
    return rec

t0 = time.time()
cand = nomis_cat.loc[nomis_cat.itl2_type, "id"].tolist()
cov = pd.DataFrame(pmap(itl2_coverage, cand))
nomis = nomis_cat.merge(cov, on="id", how="left")
nomis["itl2_n_periods"] = nomis.itl2_n_periods.fillna(0).astype(int)
print(f"done in {time.time()-t0:.0f}s | ITL2-type datasets with ITL2 data: {(nomis.itl2_n_periods > 0).sum()} of {len(cand)}")

OUT_NOMIS.parent.mkdir(parents=True, exist_ok=True)
nomis.sort_values(["itl2_n_periods","id"], ascending=[False, True]).to_csv(OUT_NOMIS, index=False)
print(f"Saved -> {OUT_NOMIS}")

  12/12
done in 10s | ITL2-type datasets with ITL2 data: 12 of 12
Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/docs/nomis_itl2_scan.csv


## A4. Results

Datasets with ITL2 data, longest ITL2 history first. `itl2_n_periods` counts periods (months, quarters or years, depending on the dataset).

In [5]:
pd.set_option("display.max_colwidth", 70)
got = nomis[nomis.itl2_n_periods > 0].sort_values("itl2_n_periods", ascending=False)
cols = ["id","name","itl2_first","itl2_last","itl2_n_periods","n_periods","status"]
print(f"{len(got)} Nomis datasets hold ITL2 (Jan 2025) data.\n")
print(got[cols].head(40).to_string(index=False))

thin = nomis[nomis.itl2_type & (nomis.itl2_n_periods == 0)]
print(f"\n{len(thin)} list the ITL2 type but have no ITL2 data in their latest periods.")
print(f"{(nomis.lad_2023 & ~nomis.itl2_type).sum()} have LA (April 2023) but no ITL2 type - LA aggregation route only.")

12 Nomis datasets hold ITL2 (Jan 2025) data.

       id                                                                      name itl2_first itl2_last  itl2_n_periods  n_periods                           status
  NM_18_1                Jobseeker's Allowance by age and duration with proportions    1992-01   2026-08             416      416.0 Current (being actively updated)
 NM_185_1 Regional gross disposable household income (GDHI) at current basic prices       1997      2024              28       28.0 Current (being actively updated)
NM_2400_1                                 UK small area gross value added estimates       1998      2023              26       26.0 Current (being actively updated)
  NM_52_1                     claimant count denominators - residents aged 16 to 64       2011      2026              16       56.0 Current (being actively updated)
   NM_1_1                          Jobseeker's Allowance with rates and proportions    2026-01   2026-08               8      519

# Part B — ONS beta API

## B1. Catalogue (paged)

In [6]:
def list_ons():
    items, offset, limit = [], 0, 100
    while True:
        d = get_json(f"{ONS}/datasets?limit={limit}&offset={offset}")
        batch = d.get("items", []) or []
        items.extend(batch)
        offset += limit
        if offset >= d.get("total_count", len(items)) or not batch:
            return items

ons_cat = list_ons()
print(f"ONS beta-API datasets: {len(ons_cat)}")

ONS beta-API datasets: 338


## B2. Scan each dataset's geography options for ITL2 codes

Geography dimensions are found by name (`geography…`) or by census area-type id (`ltla`, `utla`, `rgn`, `ctry`, `msoa`, `lsoa`, `oa` — census 2021 tables use these). The geography *code-list* link only returns metadata; the actual codes a dataset uses are its **dimension options** (`{latest_version}/dimensions/<geography>/options`, paged). ITL2 codes are 4 characters, `TL` + letter + digit (e.g. `TLC3`); ITL3 are 5 (`TLC31`).

In [7]:
ITL2 = re.compile(r"^TL[A-Z]\d$")
ITL3 = re.compile(r"^TL[A-Z]\d\d$")
CENSUS_AREAS = {"ltla", "utla", "rgn", "ctry", "msoa", "lsoa", "oa"}

def ons_options(version_href, dim_name, max_pages=20):
    codes, offset = [], 0
    for _ in range(max_pages):
        d = get_json(f"{version_href}/dimensions/{dim_name}/options?limit=1000&offset={offset}")
        codes += [x.get("option", "") for x in d.get("items", [])]
        offset += 1000
        if offset >= d.get("total_count", 0): break
    return codes

def ons_scan(ds):
    rec = {"id": ds.get("id"), "title": ds.get("title"), "geography_dim": None,
           "n_geo_codes": 0, "itl2": False, "itl3": False, "note": ""}
    lv = ((ds.get("links") or {}).get("latest_version") or {}).get("href")
    try:
        v = get_json(lv) if lv else {}
        gd = next((d for d in v.get("dimensions", []) or []
                   if "geograph" in " ".join(str(d.get(k, "")) for k in ("id", "name", "label")).lower()
                   or str(d.get("id", "")).lower() in CENSUS_AREAS), None)
        if gd is None:
            return rec
        rec["geography_dim"] = gd.get("id")
        codes = ons_options(lv, gd["name"])
        rec.update(n_geo_codes=len(codes), itl2=any(ITL2.match(c) for c in codes), itl3=any(ITL3.match(c) for c in codes))
    except Exception as e:
        rec["note"] = f"error: {e}"
    return rec

t0 = time.time()
ons = pd.DataFrame(pmap(ons_scan, ons_cat, workers=ONS_WORKERS))
ons.to_csv(OUT_ONS, index=False)
print(f"done in {time.time()-t0:.0f}s | saved -> {OUT_ONS}")
print(f"Of {len(ons)} datasets: {ons.geography_dim.notna().sum()} have a geography dimension, "
      f"{ons.itl2.sum()} at ITL2, {ons.itl3.sum()} at ITL3, errors: {(ons.note != '').sum()}")
print("Geography dimensions:", ons.geography_dim.value_counts().to_dict())
print(ons[ons.itl2][["id","title"]].to_string(index=False) if ons.itl2.any() else "  (none at ITL2)")

  338/338
done in 343s | saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/docs/ons_api_itl2_scan.csv
Of 338 datasets: 337 have a geography dimension, 0 at ITL2, 0 at ITL3, errors: 0
Geography dimensions: {'ltla': 286, 'administrative-geography': 25, 'uk-only': 15, 'countries': 4, 'nuts': 2, 'local-health-board': 1, 'parliamentary-constituencies': 1, 'enterprise-regions': 1, 'travel-to-work-area': 1, 'rgn': 1}
  (none at ITL2)
